In [ ]:
import pandas as pd
import numpy as np
from tqdm import tqdm
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.metrics import r2_score
import pickle
from sklearn.metrics import mean_squared_error
from sklearn.ensemble import RandomForestRegressor

In [ ]:
with open("data/characteristics_with_return.pkl", "rb") as f:
    data = pickle.load(f)

In [ ]:
start_year_train = 1957
start_year_test = 1987
end_year_test = 2016


predictions = []
y_test_list = []
dates = []
dic_r2_all = {}
num_coef_time = {}


for year in tqdm(range(start_year_test, end_year_test + 1), desc='Processing', unit='item'):

    start_date_train = pd.Timestamp(year=start_year_train, month=1, day=1)
    end_date_train = pd.Timestamp(year=year-13, month=12, day=31)
    start_date_val = pd.Timestamp(year=year-12, month=1, day=1)
    end_date_val = pd.Timestamp(year=year-1, month=12, day=31)   
    start_date_test = pd.Timestamp(year=year, month=1, day=1)
    end_date_test = pd.Timestamp(year=year, month=12, day=31)

   
    train_data = data[(data['DATE'] >= start_date_train) & (data['DATE'] <= end_date_train)]
    val_data = data[(data['DATE'] >= start_date_val) & (data['DATE'] <= end_date_val)]
    test_data = data[(data['DATE'] >= start_date_test) & (data['DATE'] <= end_date_test)]
    X_train = train_data.drop(['permno', 'DATE', 'RET'], axis=1).values  
    y_train = train_data['RET'].values
    X_val = val_data.drop(['permno', 'DATE', 'RET'], axis=1).values
    y_val = val_data['RET'].values
    X_test = test_data.drop(['permno', 'DATE', 'RET'], axis=1).values
    y_test = test_data['RET'].values
    
    model = RandomForestRegressor(n_estimators=50, max_depth=10, max_features='sqrt', random_state=0, n_jobs=-1)
    model.fit(X_train, y_train)
    y_val_pred = model.predict(X_val)
    y_test_pred = model.predict(X_test)
    predictions.extend(y_test_pred)
    y_test_list.extend(y_test)
    dates.append(year)
    r2 = r2_score(y_test, y_test_pred)
    dic_r2_all[year] = r2
    print(f"Year: {year}")
    print("Predictions:", y_test_pred)
    print("True Values:", y_test)
    print("R^2:", r2)


all_predictions = np.array(predictions)
all_y_test = np.array(y_test_list)
ss_res = np.sum((all_y_test - all_predictions) ** 2)  # Residual sum of squares
ss_tot = np.sum((all_y_test ) ** 2)  # Total sum of squares
total_r2_out = 1 - (ss_res / ss_tot)

print("Total Out of Sample R^2:", total_r2_out)

In [ ]:
start_year_train = 1957
start_year_test = 1987
end_year_test = 2016

predictions = []
y_test_list = []
dates = []
dic_r2_all = {}

data['DATE'] = pd.to_datetime(data['DATE'], format='%Y%m')

for year in tqdm(range(start_year_test, end_year_test + 1), desc='Processing', unit='item'):
    start_date_train = pd.Timestamp(year=start_year_train, month=1, day=1)
    end_date_train = pd.Timestamp(year=year-13, month=12, day=31)

    start_date_val = pd.Timestamp(year=year-12, month=1, day=1)
    end_date_val = pd.Timestamp(year=year-1, month=12, day=31)
    
    start_date_test = pd.Timestamp(year=year, month=1, day=1)
    end_date_test = pd.Timestamp(year=year, month=12, day=31)

    train_data = data[(data['DATE'] >= start_date_train) & (data['DATE'] <= end_date_train)]
    val_data = data[(data['DATE'] >= start_date_val) & (data['DATE'] <= end_date_val)]
    test_data = data[(data['DATE'] >= start_date_test) & (data['DATE'] <= end_date_test)]
    X_train = train_data.drop(['permno', 'DATE', 'RET'], axis=1).values  
    y_train = train_data['RET'].values
    X_val = val_data.drop(['permno', 'DATE', 'RET'], axis=1).values
    y_val = val_data['RET'].values
    X_test = test_data.drop(['permno', 'DATE', 'RET'], axis=1).values
    y_test = test_data['RET'].values
    model = RandomForestRegressor(
        n_estimators=100, 
        max_depth=20,  
        max_features='sqrt',  
        min_samples_split=5,  
        min_samples_leaf=6,  
        random_state=0,
        n_jobs=-1
    )
    model.fit(X_train, y_train)
    
    y_val_pred = model.predict(X_val)
    y_test_pred = model.predict(X_test)
    predictions.extend(y_test_pred)
    y_test_list.extend(y_test)
    dates.append(year)
    r2 = r2_score(y_test, y_test_pred)
    dic_r2_all[year] = r2
    print(f"Year: {year}")
    print("Predictions:", y_test_pred)
    print("True Values:", y_test)
    print("R^2:", r2)
all_predictions = np.array(predictions)
all_y_test = np.array(y_test_list)
ss_res = np.sum((all_y_test - all_predictions) ** 2) 
ss_tot = np.sum((all_y_test ) ** 2) 
total_r2_out = 1 - (ss_res / ss_tot)

print("Total Out of Sample R^2:", total_r2_out)